<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания N6


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Movie в C#, который будет представлять информацию о
фильмах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public interface IAwardWinning
{
    int AwardsCount { get; set; }
    void DisplayAwards();
}

public class FilmCollection<T> where T : Movie
{
    private List<T> _films = new List<T>();
    
    public void AddFilm(T film)
    {
        _films.Add(film);
        Console.WriteLine($"Фильм «{film.Title}» добавлен в коллекцию");
    }
    
    public void RemoveFilm(T film)
    {
        if (_films.Remove(film))
            Console.WriteLine($"Фильм «{film.Title}» удален из коллекции");
    }
    
    public T FindFilmByTitle(string title)
    {
        return _films.FirstOrDefault(f => 
            f.Title.Equals(title, StringComparison.OrdinalIgnoreCase));
    }
    
    public void DisplayAllFilms()
    {
        Console.WriteLine($"\nФильмы в коллекции ({typeof(T).Name}):");
        foreach (var film in _films)
        {
            Console.WriteLine($"- {film.GetInfo()}");
        }
    }
    
    public IEnumerable<T> GetFilmsByDirector(string director)
    {
        return _films.Where(f => 
            f.Director.Equals(director, StringComparison.OrdinalIgnoreCase));
    }
}

public class Review
{
    public string Author { get; set; }
    public int Rating { get; set; }
    public string Comment { get; set; }
    public DateTime ReviewDate { get; set; } 
    
    public Review(string author, int rating, string comment)
    {
        Author = author;
        Rating = rating;
        Comment = comment;
        ReviewDate = DateTime.Now;
    }
    
    public Review(string author, int rating) : this(author, rating, "Без комментария") 
    {
    }
    
    public string GetReviewInfo()
    {
        return $"{Author} ({ReviewDate:yyyy-MM-dd}): {Rating}/10 - {Comment}";
    }

    public string GetReviewInfo(bool detailed)
    {
        if (!detailed)
            return $"{Author}: {Rating}/10";
        
        return GetReviewInfo();
    }

    public bool IsRecent() 
    {
        return (DateTime.Now - ReviewDate).TotalDays <= 30;
    }
}

public class Movie
{
    private string _title;
    private int _year;
    private string _director;
    private List<Review> _reviews;
    private int _duration;
    private string _country; 
    private string _language; 

    public string Title
    {
        get => _title;
        set => _title = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Название не может быть пустым");
    }

    public int Year
    {
        get => _year;
        set => _year = (value >= 1888 && value <= DateTime.Now.Year + 2) ? value : throw new ArgumentException("Некорректный год выпуска");
    }

    public string Director
    {
        get => _director;
        set => _director = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Имя режиссера не может быть пустым");
    }

    public int Duration
    {
        get => _duration;
        set => _duration = value > 0 ? value : throw new ArgumentException("Продолжительность должна быть положительной");
    }

    public string Country
    {
        get => _country;
        set => _country = !string.IsNullOrWhiteSpace(value) ? value : "Не указана";
    }

    public string Language
    {
        get => _language;
        set => _language = !string.IsNullOrWhiteSpace(value) ? value : "Английский";
    }

    public IReadOnlyList<Review> Reviews => _reviews.AsReadOnly();

    public Movie(string title, int year, string director, int duration)
    {
        Title = title;
        Year = year;
        Director = director;
        Duration = duration;
        _reviews = new List<Review>();
        Country = "Не указана";
        Language = "Английский";
    }

    public Movie(string title, int year, string director, int duration, string country, string language) 
        : this(title, year, director, duration)
    {
        Country = country;
        Language = language;
    }

    public virtual string GetInfo()
    {
        return $"«{Title}» ({Year}), реж. {Director}, {Duration} мин, {Country}";
    }

    public string GetInfo(bool detailed)
    {
        if (!detailed)
            return $"«{Title}» ({Year}), {Director}";
        
        return GetInfo() + $", язык: {Language}";
    }

    public virtual double CalculateRating()
    {
        if (!_reviews.Any()) return 5.0;
        return _reviews.Average(r => r.Rating);
    }

    public virtual string GetAgeCategory()
    {
        int currentYear = DateTime.Now.Year;
        int age = currentYear - Year;
        
        return age switch
        {
            < 5 => "Новый",
            < 20 => "Современный",
            < 50 => "Классика",
            _ => "Старая классика"
        };
    }

    public void AddReview(Review review)
    {
        _reviews.Add(review);
        Console.WriteLine($"Добавлен отзыв для «{Title}» от {review.Author}");
    }

    public void AddReview(string author, int rating, string comment)
    {
        AddReview(new Review(author, rating, comment));
    }

    public void ShowAllReviews()
    {
        Console.WriteLine($"\nОтзывы для фильма «{Title}»:");
        foreach (var review in _reviews)
        {
            string recent = review.IsRecent() ? " (НОВЫЙ)" : "";
            Console.WriteLine($"- {review.GetReviewInfo()}{recent}");
        }
    }

    public void ShowAllReviews(bool shortVersion)
    {
        if (!shortVersion)
        {
            ShowAllReviews();
            return;
        }
        
        Console.WriteLine($"\nКраткие отзывы для «{Title}»:");
        foreach (var review in _reviews)
        {
            Console.WriteLine($"- {review.GetReviewInfo(false)}");
        }
    }

    public void RecommendSimilar(List<Movie> movies)
    {
        var similar = movies
            .Where(m => m != this && m.Director == Director)
            .ToList();

        if (similar.Any())
        {
            Console.WriteLine($"\nЕсли вам понравился «{Title}», рекомендуем:");
            foreach (var movie in similar)
            {
                Console.WriteLine($"- {movie.GetInfo()}");
            }
        }
    }

    public string GetDurationCategory() 
    {
        return Duration switch
        {
            < 60 => "Короткометражка",
            < 120 => "Среднеметражка",
            _ => "Полнометражка"
        };
    }

    public bool IsInternational()
    {
        return !string.IsNullOrEmpty(Country) && 
               !Country.Equals("США", StringComparison.OrdinalIgnoreCase) &&
               !Country.Equals("USA", StringComparison.OrdinalIgnoreCase);
    }
}

public class FeatureFilm : Movie, IAwardWinning
{
    private string _genre;
    private double _budget;
    private string _productionCompany; 
    public int AwardsCount { get; set; } 

    public string Genre
    {
        get => _genre;
        set => _genre = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Жанр не может быть пустым");
    }

    public double Budget
    {
        get => _budget;
        set => _budget = value >= 0 ? value : throw new ArgumentException("Бюджет не может быть отрицательным");
    }

    public string ProductionCompany
    {
        get => _productionCompany;
        set => _productionCompany = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public FeatureFilm(string title, int year, string director, string genre, double budget, int duration) 
        : base(title, year, director, duration)
    {
        Genre = genre;
        Budget = budget;
        AwardsCount = 0;
        ProductionCompany = "Неизвестно";
    }

    public FeatureFilm(string title, int year, string director, string genre, double budget, int duration, 
        string country, string language, string productionCompany) 
        : base(title, year, director, duration, country, language)
    {
        Genre = genre;
        Budget = budget;
        ProductionCompany = productionCompany;
        AwardsCount = 0;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", жанр: {Genre}, бюджет: ${Budget} млн";
    }

    public string GetInfo(bool includeCompany)
    {
        string info = GetInfo();
        if (includeCompany)
            info += $", студия: {ProductionCompany}";
        return info;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (Budget > 100) rating += 1.0;
        if (AwardsCount > 0) rating += AwardsCount * 0.1;
        return Math.Min(rating, 10.0);
    }

    public override string GetAgeCategory()
    {
        string baseCategory = base.GetAgeCategory();
        if (Budget > 200 && baseCategory == "Новый")
            return "Блокбастер";
        return baseCategory;
    }

    public void ShowTrailer()
    {
        Console.WriteLine($"Просмотр трейлера фильма «{Title}»");
    }

    public void ShowTrailer(string platform)
    {
        Console.WriteLine($"Просмотр трейлера фильма «{Title}» на {platform}");
    }

    public void AddToFilmFestival(List<FeatureFilm> festivalFilms)
    {
        festivalFilms.Add(this);
        Console.WriteLine($"Фильм «{Title}» добавлен в кинофестиваль!");
    }

    public void DisplayAwards() 
    {
        Console.WriteLine($"Фильм «{Title}» получил {AwardsCount} наград");
    }

    public bool IsBlockbuster()
    {
        return Budget > 100 && CalculateRating() >= 7.0;
    }
}

public class Documentary : Movie, IAwardWinning
{
    private string _topic;
    private bool _isEducational;
    private string _researchInstitution; 
    public int AwardsCount { get; set; } 

    public string Topic
    {
        get => _topic;
        set => _topic = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Тема не может быть пустой");
    }

    public bool IsEducational
    {
        get => _isEducational;
        set => _isEducational = value;
    }

    public string ResearchInstitution
    {
        get => _researchInstitution;
        set => _researchInstitution = !string.IsNullOrWhiteSpace(value) ? value : "Не указано";
    }

    public Documentary(string title, int year, string director, string topic, bool isEducational, int duration) 
        : base(title, year, director, duration)
    {
        Topic = topic;
        IsEducational = isEducational;
        AwardsCount = 0;
        ResearchInstitution = "Не указано";
    }

    public Documentary(string title, int year, string director, string topic, bool isEducational, int duration,
        string country, string language, string researchInstitution) 
        : base(title, year, director, duration, country, language)
    {
        Topic = topic;
        IsEducational = isEducational;
        ResearchInstitution = researchInstitution;
        AwardsCount = 0;
    }

    public override string GetInfo()
    {
        string edu = IsEducational ? "образовательный" : "популярный";
        return base.GetInfo() + $", тема: {Topic} ({edu})";
    }

    public string GetInfo(bool includeInstitution)
    {
        string info = GetInfo();
        if (includeInstitution && !string.IsNullOrEmpty(ResearchInstitution))
            info += $", институт: {ResearchInstitution}";
        return info;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (IsEducational) rating += 0.5;
        if (AwardsCount > 0) rating += AwardsCount * 0.2;
        return Math.Min(rating, 10.0);
    }

    public override string GetAgeCategory()
    {
        int currentYear = DateTime.Now.Year;
        int age = currentYear - Year;
        
        if (IsEducational && age <= 10)
            return "Актуальное исследование";
        
        return base.GetAgeCategory();
    }

    public void ConductInterview()
    {
        Console.WriteLine($"Проведение интервью по теме: {Topic}");
    }

    public void ConductInterview(string interviewee)
    {
        Console.WriteLine($"Проведение интервью с {interviewee} по теме: {Topic}");
    }

    public void AddToScienceConference(List<Documentary> scienceFilms)
    {
        if (IsEducational)
        {
            scienceFilms.Add(this);
            Console.WriteLine($"Документальный фильм «{Title}» добавлен в научную конференцию!");
        }
    }

    public void DisplayAwards() 
    {
        string type = IsEducational ? "Научный" : "Документальный";
        Console.WriteLine($"{type} фильм «{Title}» получил {AwardsCount} наград");
    }

    public bool HasAcademicSupport()
    {
        return IsEducational && !string.IsNullOrEmpty(ResearchInstitution) && 
               !ResearchInstitution.Equals("Не указано");
    }
}

public class HybridFilm : FeatureFilm, IAwardWinning
{
    private bool _hasDocumentaryElements;
    private int _archiveFootageMinutes; 
    public new int AwardsCount { get; set; }

    public HybridFilm(string title, int year, string director, string genre, 
        double budget, int duration, bool hasDocElements) 
        : base(title, year, director, genre, budget, duration)
    {
        _hasDocumentaryElements = hasDocElements;
        _archiveFootageMinutes = 0;
    }

    public HybridFilm(string title, int year, string director, string genre, 
        double budget, int duration, bool hasDocElements, string country, 
        string language, string productionCompany, int archiveFootage) 
        : base(title, year, director, genre, budget, duration, country, language, productionCompany)
    {
        _hasDocumentaryElements = hasDocElements;
        _archiveFootageMinutes = archiveFootage;
    }

    public int ArchiveFootageMinutes
    {
        get => _archiveFootageMinutes;
        set => _archiveFootageMinutes = value >= 0 ? value : 0;
    }

    public override string GetInfo()
    {
        string hybrid = _hasDocumentaryElements ? " (гибридный)" : "";
        string footage = _archiveFootageMinutes > 0 ? $", архивные кадры: {_archiveFootageMinutes} мин" : "";
        return base.GetInfo() + hybrid + footage;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (_hasDocumentaryElements) rating += 0.3;
        if (_archiveFootageMinutes > 10) rating += 0.2;
        return Math.Min(rating, 10.0);
    }

    public string GetFilmType()
    {
        return _hasDocumentaryElements ? "Гибридный фильм" : "Художественный фильм с элементами документального";
    }

    void IAwardWinning.DisplayAwards() 
    {
        string type = _hasDocumentaryElements ? "Гибридный" : "Художественный";
        Console.WriteLine($"{type} фильм «{Title}» собрал {AwardsCount} наград");
    }
}

        var featureCollection = new FilmCollection<FeatureFilm>();
        var documentaryCollection = new FilmCollection<Documentary>();
        var hybridCollection = new FilmCollection<HybridFilm>();

        var interstellar = new FeatureFilm("Интерстеллар", 2014, "Кристофер Нолан", 
            "фантастика", 165, 169, "США", "Английский", "Warner Bros");
        
        var earthlings = new Documentary("Земляне", 2005, "Шон Монсон", 
            "права животных", true, 95, "США", "Английский", "Animal Planet");
        
        var avatar = new HybridFilm("Аватар", 2009, "Джеймс Кэмерон", "фантастика", 
            237, 162, true, "США", "Английский", "20th Century Fox", 15);

        featureCollection.AddFilm(interstellar);
        documentaryCollection.AddFilm(earthlings);
        hybridCollection.AddFilm(avatar);

        var movies = new List<Movie> { interstellar, earthlings, avatar };

        Console.WriteLine("=== ИНФОРМАЦИЯ О ФИЛЬМАХ ===");
        foreach (var movie in movies)
        {
            Console.WriteLine(movie.GetInfo());
            Console.WriteLine($"Категория: {movie.GetDurationCategory()}");
            Console.WriteLine($"Возрастная категория: {movie.GetAgeCategory()}");
            Console.WriteLine($"Рейтинг: {movie.CalculateRating():F1}");
            Console.WriteLine($"Международный: {(movie.IsInternational() ? "Да" : "Нет")}\n");
        }


        Console.WriteLine("=== ПЕРЕГРУЗКА МЕТОДОВ ===");
        Console.WriteLine(interstellar.GetInfo());
        Console.WriteLine(interstellar.GetInfo(true));
        
        interstellar.AddReview("Критик1", 9, "Отличный фильм!");
        interstellar.AddReview(new Review("Критик2", 8));
        interstellar.ShowAllReviews();
        interstellar.ShowAllReviews(true);

        Console.WriteLine("\n=== НАГРАДЫ ===");
        var awardMovies = new List<IAwardWinning> { interstellar, earthlings, avatar };
        interstellar.AwardsCount = 5;
        earthlings.AwardsCount = 3;
        avatar.AwardsCount = 7;

        foreach (var awardMovie in awardMovies)
        {
            awardMovie.DisplayAwards();
        }

        Console.WriteLine("\n=== COLLECTIONS ===");
        featureCollection.DisplayAllFilms();
        documentaryCollection.DisplayAllFilms();
        hybridCollection.DisplayAllFilms();

        var found = featureCollection.FindFilmByTitle("Интерстеллар");
        if (found != null)
            Console.WriteLine($"Найден фильм: {found.GetInfo()}");

        Console.WriteLine("\n=== СПЕЦИФИЧЕСКИЕ МЕТОДЫ ===");
        interstellar.ShowTrailer();
        interstellar.ShowTrailer("YouTube");
        
        earthlings.ConductInterview();
        earthlings.ConductInterview("доктор наук Иванов");

        Console.WriteLine($"Аватар - тип фильма: {avatar.GetFilmType()}");
        Console.WriteLine($"Земляне имеет академическую поддержку: {earthlings.HasAcademicSupport()}");
        Console.WriteLine($"Интерстеллар блокбастер: {interstellar.IsBlockbuster()}");


Фильм «Интерстеллар» добавлен в коллекцию
Фильм «Земляне» добавлен в коллекцию
Фильм «Аватар» добавлен в коллекцию
=== ИНФОРМАЦИЯ О ФИЛЬМАХ ===
«Интерстеллар» (2014), реж. Кристофер Нолан, 169 мин, США, жанр: фантастика, бюджет: $165 млн
Категория: Полнометражка
Возрастная категория: Современный
Рейтинг: 6.0
Международный: Нет

«Земляне» (2005), реж. Шон Монсон, 95 мин, США, тема: права животных (образовательный)
Категория: Среднеметражка
Возрастная категория: Классика
Рейтинг: 5.5
Международный: Нет

«Аватар» (2009), реж. Джеймс Кэмерон, 162 мин, США, жанр: фантастика, бюджет: $237 млн (гибридный), архивные кадры: 15 мин
Категория: Полнометражка
Возрастная категория: Современный
Рейтинг: 6.5
Международный: Нет

=== ПЕРЕГРУЗКА МЕТОДОВ ===
«Интерстеллар» (2014), реж. Кристофер Нолан, 169 мин, США, жанр: фантастика, бюджет: $165 млн
«Интерстеллар» (2014), реж. Кристофер Нолан, 169 мин, США, жанр: фантастика, бюджет: $165 млн, студия: Warner Bros
Добавлен отзыв для «Интерстеллар» от Крити